# 🤖 ML Pipeline for Vehicle Price Prediction

This section includes:
- Feature engineering
- Model training (Random Forest & XGBoost)
- Model evaluation
- Model persistence
- Docker-ready deployment

In [2]:
# ===================================
# 1. Feature Engineering
# ===================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
df = pd.read_csv('clean#1.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Handle missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Drop rows with missing critical features
df_ml = df.dropna(subset=['Price', 'Year', 'Milleage']).copy()

print(f"\nDataset shape after dropping NaN: {df_ml.shape}")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# ===================================
# 2. Encode Categorical Features
# ===================================

# Fill missing categorical values
df_ml['Vehicle Type'] = df_ml['Vehicle Type'].fillna('Unknown')
df_ml['Make'] = df_ml['Make'].fillna('Unknown')
df_ml['Model'] = df_ml['Model'].fillna('Unknown')
df_ml['District'] = df_ml['District'].fillna('Unknown')
df_ml['Condition'] = df_ml['Condition'].fillna('Used')

# Create label encoders
encoders = {}
categorical_cols = ['Vehicle Type', 'Make', 'Model', 'District', 'Condition']

for col in categorical_cols:
    le = LabelEncoder()
    df_ml[f'{col}_encoded'] = le.fit_transform(df_ml[col].astype(str))
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique values")

# Save encoders for later use
joblib.dump(encoders, 'label_encoders.pkl')
print("\n✅ Label encoders saved!")

# Feature engineering: Vehicle age
current_year = datetime.now().year
df_ml['Vehicle_Age'] = current_year - df_ml['Year']

# Price per year (depreciation indicator)
df_ml['Price_per_Year'] = df_ml['Price'] / (df_ml['Vehicle_Age'] + 1)

# Mileage per year
df_ml['Mileage_per_Year'] = df_ml['Milleage'] / (df_ml['Vehicle_Age'] + 1)

print(f"\n✅ Feature engineering completed!")
print(f"New features: Vehicle_Age, Price_per_Year, Mileage_per_Year")

In [ ]:
# ===================================
# 3. Prepare Train/Test Split
# ===================================

# Select features for modeling
feature_cols = [
    'Vehicle Type_encoded', 'Make_encoded', 'Model_encoded',
    'District_encoded', 'Condition_encoded', 'Year', 'Milleage',
    'Vehicle_Age', 'Mileage_per_Year'
]

X = df_ml[feature_cols].copy()
y = df_ml['Price'].copy()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, 'scaler.pkl')

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"✅ Data split and scaling completed!")

In [ ]:
# ===================================
# 4. Train Random Forest Model
# ===================================

print("🌲 Training Random Forest Regressor...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test_scaled)

# Evaluation metrics
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print("\n🎯 Random Forest Results:")
print(f"  MAE:  Rs. {rf_mae:,.2f}")
print(f"  RMSE: Rs. {rf_rmse:,.2f}")
print(f"  R² Score: {rf_r2:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top 5 Important Features:")
print(feature_importance.head())

In [ ]:
# ===================================
# 5. Train XGBoost Model
# ===================================

print("🚀 Training XGBoost Regressor...")
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

xgb_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test_scaled)

# Evaluation metrics
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
xgb_r2 = r2_score(y_test, y_pred_xgb)

print("\n🎯 XGBoost Results:")
print(f"  MAE:  Rs. {xgb_mae:,.2f}")
print(f"  RMSE: Rs. {xgb_rmse:,.2f}")
print(f"  R² Score: {xgb_r2:.4f}")

# Comparison
print("\n📊 Model Comparison:")
print(f"{'Model':<15} {'MAE':>15} {'RMSE':>15} {'R²':>10}")
print("-" * 60)
print(f"{'Random Forest':<15} Rs. {rf_mae:>12,.2f} Rs. {rf_rmse:>12,.2f} {rf_r2:>9.4f}")
print(f"{'XGBoost':<15} Rs. {xgb_mae:>12,.2f} Rs. {xgb_rmse:>12,.2f} {xgb_r2:>9.4f}")

# Select best model
if xgb_r2 > rf_r2:
    best_model = xgb_model
    best_model_name = "XGBoost"
    best_metrics = {'MAE': xgb_mae, 'RMSE': xgb_rmse, 'R2': xgb_r2}
else:
    best_model = rf_model
    best_model_name = "Random Forest"
    best_metrics = {'MAE': rf_mae, 'RMSE': rf_rmse, 'R2': rf_r2}

print(f"\n✅ Best model: {best_model_name}")

In [ ]:
# ===================================
# 6. Save Models and Metadata
# ===================================

# Create model directory if it doesn't exist
import os
model_dir = '../../ML_Model'
os.makedirs(model_dir, exist_ok=True)

# Save both models
joblib.dump(rf_model, f'{model_dir}/random_forest_model.pkl')
joblib.dump(xgb_model, f'{model_dir}/xgboost_model.pkl')
joblib.dump(best_model, f'{model_dir}/best_model.pkl')

# Move encoders and scaler to model directory
import shutil
shutil.copy('label_encoders.pkl', f'{model_dir}/label_encoders.pkl')
shutil.copy('scaler.pkl', f'{model_dir}/scaler.pkl')

# Save model metadata
metadata = {
    'best_model': best_model_name,
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'features': feature_cols,
    'metrics': {
        'random_forest': {
            'MAE': float(rf_mae),
            'RMSE': float(rf_rmse),
            'R2': float(rf_r2)
        },
        'xgboost': {
            'MAE': float(xgb_mae),
            'RMSE': float(xgb_rmse),
            'R2': float(xgb_r2)
        },
        'best_model': best_metrics
    },
    'data_stats': {
        'training_samples': int(len(X_train)),
        'test_samples': int(len(X_test)),
        'total_samples': int(len(df_ml))
    }
}

with open(f'{model_dir}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"✅ Models saved to {model_dir}/")
print(f"   - random_forest_model.pkl")
print(f"   - xgboost_model.pkl")
print(f"   - best_model.pkl ({best_model_name})")
print(f"   - label_encoders.pkl")
print(f"   - scaler.pkl")
print(f"   - model_metadata.json")
print(f"\n🎉 ML Pipeline completed successfully!")